# Tutorial: Runtime project smoke

Audience:
- Desarrollo de AMDockVS.

Prerequisites:
- Ejecutar el notebook desde un entorno donde `AMDockVS` y `MolSuite` sean importables.

Learning goals:
- Verificar que `AMDockVSRuntime` solo expone runtime/orquestación y fachadas de dominio.
- Validar creación de proyecto y recursos básicos.


## Outline

1. Setup del repositorio y runtime
2. Crear un proyecto temporal
3. Verificar fachadas y ausencia de métodos directos viejos


In [1]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
AMDOCK_SRC = REPO_ROOT / "src"
MOLSUITE_SRC = REPO_ROOT.parent / "MolSuite" / "src"
for path in (AMDOCK_SRC, MOLSUITE_SRC):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from amdockvs.runtime import AMDockVSRuntime

TMP_ROOT = REPO_ROOT / "notebooks" / "_tmp"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="01_runtime_project_", dir=TMP_ROOT))

runtime = AMDockVSRuntime()
project = runtime.create_project(
    name=f"runtime-smoke-{PROJECT_DIR.name}",
    folder=PROJECT_DIR,
    description="Notebook smoke test for runtime structure.",
)

project


[2026-04-29 07:54:13,126] [  INFO  ] amdockvs.app.lifecycle: Logging manager started


[2026-04-29 07:54:13,198] [  INFO  ] amdockvs.executor.core: Registered thread executor 'thread' (max_workers=32)


[2026-04-29 07:54:13,200] [  INFO  ] amdockvs.executor.core: Registered process executor 'process' (dynamic CPU allocation)


[2026-04-29 07:54:13,203] [  INFO  ] amdockvs.executor.core: Executor manager started (total_cpu=16)


[2026-04-29 07:54:13,203] [  INFO  ] amdockvs.app.core: MolSuite runtime listo: executor=/home/mario/PycharmProjects/AMDockVS/notebooks/_tmp/01_runtime_project_czw0lb0m/executor.db


[2026-04-29 07:54:13,274] [  INFO  ] amdockvs.project.core: Project context active: id=0e6eec25-ef98-459f-8290-ca0df3d1c549 path=/home/mario/PycharmProjects/AMDockVS/notebooks/_tmp/01_runtime_project_czw0lb0m [project_id=0e6eec25-ef98-459f-8290-ca0df3d1c549]


[2026-04-29 07:54:13,274] [  INFO  ] amdockvs.app.core: Project runtime active for project=0e6eec25-ef98-459f-8290-ca0df3d1c549 [project_id=0e6eec25-ef98-459f-8290-ca0df3d1c549]


[2026-04-29 07:54:13,277] [  INFO  ] amdockvs.app.core: Project created and activated: id=0e6eec25-ef98-459f-8290-ca0df3d1c549 name=runtime-smoke-01_runtime_project_czw0lb0m [project_id=0e6eec25-ef98-459f-8290-ca0df3d1c549]


[2026-04-29 07:54:13,301] [  INFO  ] molsuite.app.lifecycle: Logging manager started


ProjectSummary(id='0e6eec25-ef98-459f-8290-ca0df3d1c549', name='runtime-smoke-01_runtime_project_czw0lb0m', path=PosixPath('/home/mario/PycharmProjects/AMDockVS/notebooks/_tmp/01_runtime_project_czw0lb0m'), app_id='amdockvs', scope='docking', description='Notebook smoke test for runtime structure.', tags=[], favorite=False, created_at=datetime.datetime(2026, 4, 29, 7, 54, 13, 155370), updated_at=datetime.datetime(2026, 4, 29, 7, 54, 13, 276461))

## Runtime surface

La expectativa actual es que el runtime publique proyecto/jobs y las fachadas de dominio, pero no métodos directos de negocio como `import_ligands()` o `run_docking()`.


In [2]:
project_paths = runtime.get_project_paths()
assert project_paths["project_root"] == PROJECT_DIR.resolve()
assert project_paths["ligand_data_dir"].exists()
assert project_paths["receptor_data_dir"].exists()
assert project_paths["jobs_dir"].exists()

for attr in ("loader", "ligands", "receptors", "qsar", "docking"):
    assert hasattr(runtime, attr), attr

legacy_methods = [
    "import_ligands",
    "import_receptors",
    "calculate_descriptors",
    "run_docking",
    "run_screening",
    "list_results",
]
missing = [name for name in legacy_methods if hasattr(runtime, name)]
assert not missing, f"Legacy runtime methods still present: {missing}"

{
    "project_id": project.id,
    "project_root": str(project_paths["project_root"]),
    "available_facades": ["loader", "ligands", "receptors", "qsar", "docking"],
}


{'project_id': '0e6eec25-ef98-459f-8290-ca0df3d1c549',
 'project_root': '/home/mario/PycharmProjects/AMDockVS/notebooks/_tmp/01_runtime_project_czw0lb0m',
 'available_facades': ['loader', 'ligands', 'receptors', 'qsar', 'docking']}

In [3]:
runtime.get_executor_capability_matrix()


{'thread': {'backend': 'thread',
  'mode': 'local',
  'support_level': 'stable',
  'shared_filesystem': True,
  'consumes_local_cpu_tokens': False,
  'supports_inline': True,
  'supports_bytes': True,
  'supports_file_input': True,
  'supports_db_input': True,
  'shared_fs': True,
  'local_resource_accounting': 'none'},
 'process': {'backend': 'process',
  'mode': 'local',
  'support_level': 'stable',
  'shared_filesystem': True,
  'consumes_local_cpu_tokens': True,
  'supports_inline': True,
  'supports_bytes': True,
  'supports_file_input': True,
  'supports_db_input': True,
  'shared_fs': True,
  'local_resource_accounting': 'dynamic'}}